In [ ]:
from pathlib import Path
import sys

search_locations = [Path.cwd(), *Path.cwd().parents]

repo_root = next(
    (
        path
        for path in search_locations
        if (path / "src" / "enares").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("Could not locate the repository root")

src_path = str(repo_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Repository:", repo_root.name)

In [ ]:
from enares.config import load_config

config = load_config()

print("Project:", config.project_id)
print("Location:", config.location)
print("Raw dataset:", config.datasets.raw)
print("Cleaned dataset:", config.datasets.cleaned)
print("Analytical dataset:", config.datasets.analytical)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(
    project=config.project_id,
    location=config.location,
)

available_datasets = {
    dataset.dataset_id
    for dataset in client.list_datasets()
}

required_datasets = {
    config.datasets.raw,
    config.datasets.cleaned,
    config.datasets.analytical,
    config.datasets.outputs,
}

missing_datasets = required_datasets - available_datasets

assert not missing_datasets, (
    f"Missing datasets: {sorted(missing_datasets)}"
)

print("Dataset validation: PASS")
print(sorted(required_datasets))

In [ ]:
raw_dataset_id = (
    f"{config.project_id}.{config.datasets.raw}"
)

available_tables = {
    table.table_id
    for table in client.list_tables(raw_dataset_id)
}

required_tables = {
    "raw_crs04_cap100",
    "raw_crs04_cap200",
    "raw_crs04_cap248",
    "raw_crs04_cap300",
}

missing_tables = required_tables - available_tables

assert not missing_tables, (
    f"Missing raw tables: {sorted(missing_tables)}"
)

print("Raw table validation: PASS")
print(sorted(required_tables))